# 52. mIoU와 Class별 성능 비교

baseline 결과를 class별로 쪼개서 봅니다. 배경 class가 많으면 pixel accuracy는 높아도 객체 class IoU는 낮을 수 있으므로, 이후 실험 비교의 기준 metric은 `mean IoU`와 `class IoU`로 둡니다.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "seg7_utils.py").exists():
    NOTEBOOK_DIR = Path("Vision 기초/7장")

sys.path.append(str(NOTEBOOK_DIR))
DATA_ROOT = NOTEBOOK_DIR / "data" / "mini_shapes_seg"
RUNS_ROOT = NOTEBOOK_DIR / "runs"

from seg7_utils import *
set_korean_font()
set_seed(7)

## 52-1. Baseline metrics 불러오기

In [ ]:
baseline_metrics = read_json(RUNS_ROOT / "50_baseline_fcn" / "metrics.json")
confusion = np.array(baseline_metrics["confusion_matrix"])
class_iou = baseline_metrics["class_iou"]

print("pixel_accuracy:", baseline_metrics["pixel_accuracy"])
print("mean_iou:", baseline_metrics["mean_iou"])
class_iou

## 52-2. Confusion matrix와 class IoU 시각화

In [ ]:
import matplotlib.pyplot as plt

run_dir = RUNS_ROOT / "52_metric_baseline"
(run_dir / "pred_samples").mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
im = axes[0].imshow(confusion, cmap="Blues")
axes[0].set_title("confusion matrix")
axes[0].set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=30, ha="right")
axes[0].set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
for y in range(confusion.shape[0]):
    for x in range(confusion.shape[1]):
        axes[0].text(x, y, int(confusion[y, x]), ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=axes[0], fraction=0.046)

axes[1].bar(class_iou.keys(), class_iou.values(), color=["#94a3b8", "#ef4444", "#3b82f6"])
axes[1].set_ylim(0, 1)
axes[1].set_title("class IoU")
axes[1].grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(run_dir / "pred_samples" / "baseline_metric_summary.png", dpi=140)
plt.show()

summary_metrics = dict(baseline_metrics)
summary_metrics["experiment_name"] = "52_metric_baseline"
save_json(run_dir / "config.json", {
    "experiment_name": "52_metric_baseline",
    "source_run": "50_baseline_fcn",
    "note": "class-wise metric analysis for baseline",
})
save_json(run_dir / "metrics.json", summary_metrics)